# [Baseline] 3-Stage 영상 분석 모델 — 추론 및 제출 ZIP 생성

학습 노트북이 저장한 Stage별 모델을 불러와 평가 데이터를 예측하고, 실제 제출 가능한 `submit.zip`을 생성합니다.

이 대회는 결과 CSV를 직접 제출하는 방식이 아니라 **학습된 모델과 추론 코드를 ZIP으로 제출하는 코드 제출 대회**입니다. 평가 환경에서는 `inference.py`의 아래 세 함수를 호출합니다.

- `predict_stage1(data_dir, model_dir)` : 재녹화 여부 판별
- `predict_stage2(data_dir, model_dir)` : 충돌·진입 프레임, 회피 공간 여부, 진입 방향 예측
- `predict_stage3(data_dir, model_dir)` : 0.1초 단위 가감속·조향 범주 예측

제출 파일 구조는 다음과 같습니다.

```text
submit.zip
├── model/
│   ├── stage1/
│   │   └── best.pt
│   ├── stage2/
│   │   ├── best.pt
│   │   └── resnet18-f37072fd.pth
│   └── stage3/
│       └── best.pt
├── inference.py
└── requirements.txt
```

배포된 `data/`는 코드 실행과 입출력 형식 확인을 위한 소규모 공개 예제입니다. 실제 평가 데이터는 제출 ZIP에 포함하지 않으며, 평가 시 동일한 Stage별 경로 구조로 제공됩니다.

Stage 1의 재녹화 클래스 예제는 제출 구조 확인을 위해 재녹화 과정에서 발생할 수 있는 영상 특성을 모사한 파생 예제이며, 실제 다른 기기로 재촬영한 데이터가 아닙니다.

Stage 2 공개 예제는 충돌시점 라벨만을 활용하여 모델을 학습합니다. 진입시점·회피 공간 여부·진입 방향에 대한 정답 라벨은 포함되어 있지 않으나, 전체 제출 인터페이스와 추론 코드의 실행 구조를 확인할 수 있도록 모델의 초기 출력값을 이용하여 해당 항목의 예측 결과를 생성합니다. 따라서 충돌시점을 제외한 세 항목의 출력은 유효한 학습 성능을 나타내지 않으며, 참가자는 별도로 확보한 학습데이터와 모델을 활용하여 해당 항목의 예측 로직을 구현해야 합니다.

Stage 3 공개 예제는 코드 실행 확인을 위한 소규모·희소 라벨 데이터이며, 실제 대회 성능 확보를 위해서는 참가자가 별도의 학습데이터와 라벨링 전략을 구성해야 합니다.

## 1. 라이브러리와 공통 설정

영상 로딩에는 OpenCV, 이미지 전처리에는 Pillow·torchvision, 추론에는 PyTorch를 사용합니다. 학습 때 적용한 크기 조정과 정규화를 추론에서도 동일하게 적용해야 합니다.

아래 Stage별 코드 셀을 실행한 뒤, 5번 단계에서 해당 셀들을 하나로 결합하여 참가자가 제출할 `inference.py`를 생성합니다.


In [ ]:
# BASELINE_INFERENCE_PART
"""3-Stage 영상 분석 베이스라인 추론 코드.

각 Stage의 평가 데이터를 예측하여 정해진 형식의 DataFrame을 반환한다.
"""
from __future__ import annotations

import re
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.video import mvit_v2_s

VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".3gp", ".3gpp", ".wmv"}
ACCEL = ["ACCELERATING", "DECELERATING", "CONSTANT", "STOPPED"]
STEER = ["LEFT", "STRAIGHT", "RIGHT"]
S1_MEAN = torch.tensor([0.45, 0.45, 0.45])[:, None, None, None]
S1_STD = torch.tensor([0.225, 0.225, 0.225])[:, None, None, None]
S3_MEAN = torch.tensor([0.45, 0.45, 0.45])[:, None, None]
S3_STD = torch.tensor([0.225, 0.225, 0.225])[:, None, None]
cv2.setNumThreads(1)


def _device() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError("이 제출물은 CUDA GPU 평가환경을 필요로 합니다.")
    return torch.device("cuda")


def _video_paths(root: Path):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXT)




## 2. Stage 1 — 재녹화 여부 판별

Stage 1은 각 영상을 여러 구간으로 나누어 16프레임 클립을 구성하고, 학습 노트북에서 저장한 MViTv2-S 모델로 원본(`ORIGINAL`)과 재녹화(`RERECORDED`)를 예측합니다. 여러 클립의 로짓을 평균하여 영상별 최종 클래스를 결정합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `answer` | `ORIGINAL` 또는 `RERECORDED` |


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 1: MViTv2-S 기반 재녹화 분류기
# ---------------------------------------------------------------------------
def _clip_ids(path: Path, n: int, slot: int, slots: int):
    cap = cv2.VideoCapture(str(path))
    total = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
    center = (slot + 0.5) * total / slots
    start = max(0, min(total - n, round(center - n / 2)))
    return np.linspace(start, min(total - 1, start + n - 1), n).round().astype(int)


def _decode_stage1_clip(path: Path, size: int, frame_ids):
    cap = cv2.VideoCapture(str(path))
    out = []
    wanted = [int(x) for x in frame_ids]
    cap.set(cv2.CAP_PROP_POS_FRAMES, wanted[0])
    pos = wanted[0]
    for idx in wanted:
        ok = False
        bgr = None
        while pos <= idx:
            ok, bgr = cap.read()
            pos += 1
            if not ok:
                break
        if not ok or bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        scale = size / min(h, w)
        nh, nw = max(size, round(h * scale)), max(size, round(w * scale))
        rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
        y, x = (nh - size) // 2, (nw - size) // 2
        out.append(rgb[y : y + size, x : x + size])
    cap.release()
    if not out:
        raise ValueError(f"cannot decode video: {path.name}")
    while len(out) < len(wanted):
        out.append(out[-1])
    x = torch.from_numpy(np.stack(out)).permute(3, 0, 1, 2).float() / 255.0
    return (x - S1_MEAN) / S1_STD


class _Stage1Clips(Dataset):
    def __init__(self, videos, slots, size, frames):
        self.videos, self.slots, self.size, self.frames = videos, slots, size, frames

    def __len__(self):
        return len(self.videos) * self.slots

    def __getitem__(self, index):
        video_index, slot = index // self.slots, index % self.slots
        path = self.videos[video_index]
        try:
            x = _decode_stage1_clip(path, self.size, _clip_ids(path, self.frames, slot, self.slots))
            valid = 1
        except Exception:
            x = torch.zeros(3, self.frames, self.size, self.size)
            valid = 0
        return x, video_index, valid


def predict_stage1(data_dir, model_dir):
    device = _device()
    checkpoint = torch.load(Path(model_dir) / "best.pt", map_location="cpu", weights_only=False)
    size, frames = int(checkpoint["size"]), int(checkpoint["frames"])
    model = mvit_v2_s(weights=None)
    model.head[1] = nn.Linear(model.head[1].in_features, 2)
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()

    root = Path(data_dir) / "videos"
    videos = _video_paths(root)
    slots = 3
    dataset = _Stage1Clips(videos, slots, size, frames)
    loader = DataLoader(dataset, batch_size=4, num_workers=4, pin_memory=True)
    scores = [[] for _ in videos]
    with torch.inference_mode():
        for clips, video_indices, valid in loader:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                prob = torch.softmax(model(clips.to(device, non_blocking=True)), 1)[:, 1]
            for idx, value, ok in zip(video_indices.tolist(), prob.float().cpu().tolist(), valid.tolist()):
                if ok:
                    scores[idx].append(float(value))

    rows = []
    for path, values in zip(videos, scores):
        probability = float(np.mean(values)) if values else 1.0
        rows.append({"ID": path.stem, "answer": "RERECORDED" if probability >= 0.5 else "ORIGINAL"})
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(rows, columns=["ID", "answer"])




## 3. Stage 2 — 사고 주요시점·상황 예측

Stage 2는 영상별 프레임 이미지를 순서대로 불러와 ResNet18로 특징을 추출하고, 양방향 GRU로 시간 흐름을 분석합니다. 충돌 프레임과 진입 프레임은 시점별 점수가 가장 높은 프레임으로 선택하고, 회피 공간 여부와 피해차량 진입 방향은 분류 헤드의 출력으로 결정합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `collision_frame` | 충돌 시점의 원본 프레임 번호 |
| `entry_frame` | 피해차량이 피의차량 차선에 최초 진입한 원본 프레임 번호 |
| `evasion_space` | 충돌 당시 회피 공간 여부 |
| `entry_side` | 블랙박스 영상 기준 피해차량 진입 방향 |

프레임 파일명에 포함된 번호를 원본 프레임 번호로 그대로 사용하므로, 이미지 순번을 새로 매기지 않습니다.


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 2: ResNet18 + BiGRU 기반 네 과업 모델
# ---------------------------------------------------------------------------
class _Stage2Frames(Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


class _Stage2Temporal(nn.Module):
    def __init__(self):
        super().__init__()
        self.r = nn.GRU(512, 192, 2, batch_first=True, bidirectional=True, dropout=0.15)
        self.tc = nn.Linear(384, 1)
        self.te = nn.Linear(384, 1)
        self.scene = nn.Sequential(nn.Linear(768, 192), nn.ReLU(), nn.Dropout(0.2), nn.Linear(192, 4))

    def forward(self, x):
        h, _ = self.r(x)
        collision_logits = self.tc(h).squeeze(-1)
        entry_logits = self.te(h).squeeze(-1)
        collision_index = collision_logits.argmax(1)
        entry_index = entry_logits.argmax(1)
        batch = torch.arange(len(h), device=h.device)
        scene_input = torch.cat([h[batch, collision_index], h[batch, entry_index]], 1)
        return collision_index, entry_index, self.scene(scene_input)


def _frame_number(path: Path):
    match = re.search(r"(\d+)$", path.stem)
    return int(match.group(1)) if match else 0


def predict_stage2(data_dir, model_dir):
    device = _device()
    model_dir = Path(model_dir)
    transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
    backbone = resnet18(weights=None)
    backbone.load_state_dict(torch.load(model_dir / "resnet18-f37072fd.pth", map_location="cpu", weights_only=True))
    backbone.fc = nn.Identity()
    backbone.to(device).eval()
    temporal = _Stage2Temporal()
    temporal.load_state_dict(torch.load(model_dir / "best.pt", map_location="cpu", weights_only=False)["model"])
    temporal.to(device).eval()

    image_root = Path(data_dir) / "images"
    folders = sorted(p for p in image_root.iterdir() if p.is_dir())
    rows = []
    with torch.inference_mode():
        for folder in folders:
            # 평가 정의상 모든 원본 프레임을 사용한다(stride=1).
            paths = sorted(
                (p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}),
                key=_frame_number,
            )
            if not paths:
                continue
            loader = DataLoader(_Stage2Frames(paths, transform), batch_size=256, num_workers=6, pin_memory=True)
            features = []
            for images in loader:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    features.append(backbone(images.to(device, non_blocking=True)).float().cpu())
            sequence = torch.cat(features)[None].to(device)
            collision_idx, entry_idx, scene = temporal(sequence)
            frame_numbers = [_frame_number(path) for path in paths]
            rows.append(
                {
                    "ID": folder.name,
                    "collision_frame": frame_numbers[int(collision_idx)],
                    "entry_frame": frame_numbers[int(entry_idx)],
                    "evasion_space": int(scene[:, :2].argmax(1)),
                    "entry_side": "RIGHT" if int(scene[:, 2:].argmax(1)) else "LEFT",
                }
            )
    del backbone, temporal
    torch.cuda.empty_cache()
    return pd.DataFrame(
        rows, columns=["ID", "collision_frame", "entry_frame", "evasion_space", "entry_side"]
    )




## 4. Stage 3 — 차량 거동 특성 범주 예측

Stage 3은 10Hz 실차 영상을 프레임 단위로 읽고, MViTv2-S 공유 백본과 두 개의 분류 헤드로 가감속 4개 범주와 조향 3개 범주를 예측합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `sample_index` | 0.1초 단위 표본 순번 |
| `accel_label` | `ACCELERATING`, `DECELERATING`, `CONSTANT`, `STOPPED` |
| `steer_label` | `LEFT`, `STRAIGHT`, `RIGHT` |


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 3: MViTv2-S 기반 가감속/조향 다중헤드 모델
# ---------------------------------------------------------------------------
class _Stage3MViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = mvit_v2_s(weights=None)
        dimension = self.backbone.head[1].in_features
        self.backbone.head = nn.Identity()
        self.accel = nn.Linear(dimension, 4)
        self.steer = nn.Linear(dimension, 3)

    def forward(self, x):
        features = self.backbone(x)
        return self.accel(features), self.steer(features)


def _stage3_frames(path: Path):
    capture = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, bgr = capture.read()
        if not ok:
            break
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(rgb)
        width, height = image.size
        scale = 256 / min(width, height)
        image = image.resize((round(width * scale), round(height * scale)))
        width, height = image.size
        x, y = (width - 224) // 2, (height - 224) // 2
        image = image.crop((x, y, x + 224, y + 224))
        frames.append(torch.from_numpy(np.asarray(image).copy()).permute(2, 0, 1).to(torch.uint8))
    capture.release()
    if not frames:
        raise ValueError(f"cannot decode video: {path.name}")
    return torch.stack(frames)


def predict_stage3(data_dir, model_dir):
    device = _device()
    checkpoint = torch.load(Path(model_dir) / "best.pt", map_location="cpu", weights_only=False)
    model = _Stage3MViT()
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()
    videos = _video_paths(Path(data_dir) / "videos")
    rows = []
    with torch.inference_mode():
        for path in videos:
            frames = _stage3_frames(path)
            count = len(frames)
            centers = np.arange(count)
            accel_predictions, steer_predictions = [], []
            for start in range(0, count, 8):
                center = centers[start : start + 8]
                indices = np.clip(center[:, None] - 8 + np.arange(16)[None, :], 0, count - 1)
                clips = frames[torch.from_numpy(indices)].permute(0, 2, 1, 3, 4).float() / 255.0
                clips = (clips - S3_MEAN[None, :, None, :, :]) / S3_STD[None, :, None, :, :]
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    accel_logits, steer_logits = model(clips.to(device, non_blocking=True))
                accel_predictions.extend(accel_logits.argmax(1).cpu().tolist())
                steer_predictions.extend(steer_logits.argmax(1).cpu().tolist())
            for sample_index, (accel, steer) in enumerate(zip(accel_predictions, steer_predictions)):
                rows.append(
                    {
                        "ID": path.stem,
                        "sample_index": sample_index,
                        "accel_label": ACCEL[accel],
                        "steer_label": STEER[steer],
                    }
                )
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(rows, columns=["ID", "sample_index", "accel_label", "steer_label"])


## 5. `inference.py` 생성 및 함수 검사

앞에서 작성한 공통 설정과 Stage별 추론 코드를 순서대로 결합하여 `./inference.py`를 생성합니다.

생성 직후 Python 문법과 `predict_stage1`, `predict_stage2`, `predict_stage3` 함수의 존재 여부를 검사합니다. 따라서 이 단계가 정상적으로 완료된 `inference.py`만 이후 공개 예제 추론과 제출 ZIP 생성에 사용됩니다.


In [ ]:
import ast
import importlib.util
import json

ROOT = Path.cwd()
NOTEBOOK_PATH = ROOT / "[Baseline_Inference]_3Stage_추론및ZIP생성.ipynb"
INFERENCE_PATH = ROOT / "inference.py"

if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(f"현재 추론 노트북을 찾을 수 없습니다: {NOTEBOOK_PATH}")

notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
inference_parts = []
inference_marker = "# " + "BASELINE_INFERENCE_PART"
for cell in notebook["cells"]:
    if cell.get("cell_type") != "code":
        continue
    cell_source = "".join(cell.get("source", []))
    if inference_marker in cell_source:
        inference_parts.append(cell_source)

if len(inference_parts) != 4:
    raise RuntimeError(
        "inference.py 생성에 필요한 코드 셀은 4개여야 합니다. "
        f"현재 확인된 셀: {len(inference_parts)}개. 새 배포본의 추론 노트북을 저장한 뒤 다시 실행해 주세요."
    )

inference_source = "\n\n".join(part.rstrip() for part in inference_parts) + "\n"
tree = ast.parse(inference_source, filename="inference.py")
defined_functions = {
    node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
required_functions = {"predict_stage1", "predict_stage2", "predict_stage3"}
missing_functions = sorted(required_functions - defined_functions)
if missing_functions:
    raise RuntimeError("필수 추론 함수가 없습니다: " + str(missing_functions))

INFERENCE_PATH.write_text(inference_source, encoding="utf-8")

# 이후 예제 추론도 방금 생성한 inference.py를 직접 불러와 수행합니다.
spec = importlib.util.spec_from_file_location("baseline_inference", INFERENCE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"inference.py를 불러올 수 없습니다: {INFERENCE_PATH}")
baseline_inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline_inference)

print("생성 완료:", INFERENCE_PATH)
print("확인된 함수:", sorted(required_functions))


## 6. 모델 파일 및 공개 예제 데이터 확인

학습 노트북을 먼저 실행하면 `./model/stage1`, `./model/stage2`, `./model/stage3`에 체크포인트가 저장됩니다. 아래 셀은 모델 파일과 공개 예제 데이터의 존재 여부를 확인합니다.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
MODEL_DIR = ROOT / "model"
DATA_DIR = ROOT / "data"

required_models = [
    MODEL_DIR / "stage1" / "best.pt",
    MODEL_DIR / "stage2" / "best.pt",
    MODEL_DIR / "stage2" / "resnet18-f37072fd.pth",
    MODEL_DIR / "stage3" / "best.pt",
]
missing = [str(path) for path in required_models if not path.is_file()]
if missing:
    raise FileNotFoundError("학습 노트북을 먼저 실행해 주세요: " + str(missing))

for stage in ("stage1", "stage2", "stage3"):
    path = DATA_DIR / stage
    if not path.exists():
        raise FileNotFoundError(f"공개 예제 데이터 폴더가 없습니다: {path}")

print("모델과 공개 예제 데이터 확인 완료")


## 7. 공개 예제 평가 입력 생성

배포 데이터는 학습용 원천 구조이므로 아래 셀에서 실제 평가 입력과 같은 구조의 임시 폴더를 만듭니다. Stage 2는 원본 프레임을 모두 JPEG로 변환하며, 프레임 번호는 0부터 시작합니다.

이 과정은 공개 예제 실행을 위한 준비 단계입니다. 실제 평가 시에는 준비된 평가 입력이 각 함수의 `data_dir`로 전달됩니다.


In [ ]:
import shutil
import cv2

SMOKE_DIR = ROOT / "sample_evaluation_data"
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

(SMOKE_DIR / "stage1" / "videos").mkdir(parents=True)
(SMOKE_DIR / "stage2" / "images").mkdir(parents=True)
(SMOKE_DIR / "stage3" / "videos").mkdir(parents=True)

# Stage 1 공개 예제
for label, folder in [("O", "original"), ("R", "rerecorded")]:
    for index, path in enumerate(sorted((DATA_DIR / "stage1" / folder).glob("*")), 1):
        target = SMOKE_DIR / "stage1" / "videos" / f"SAMPLE_S1_{label}_{index:03d}{path.suffix.lower()}"
        shutil.copy2(path, target)

# Stage 2 공개 예제: 원본 프레임 번호를 유지하며 전체 프레임 추출
for index, video_path in enumerate(sorted((DATA_DIR / "stage2" / "videos").glob("*")), 1):
    frame_dir = SMOKE_DIR / "stage2" / "images" / f"SAMPLE_S2_{index:03d}"
    frame_dir.mkdir()
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    while True:
        ok, image = capture.read()
        if not ok:
            break
        cv2.imwrite(str(frame_dir / f"frame_{frame_index:06d}.jpg"), image)
        frame_index += 1
    capture.release()

# Stage 3 공개 예제
for path in sorted((DATA_DIR / "stage3" / "videos").glob("*")):
    shutil.copy2(path, SMOKE_DIR / "stage3" / "videos" / path.name)

print("예제 평가 입력 생성 완료:", SMOKE_DIR)


## 8. 생성한 `inference.py`로 Stage별 예측

각 함수는 `pandas.DataFrame`을 반환합니다. 아래 셀은 공개 예제를 예측하고 결과 형식을 확인하기 위해 Stage별 CSV를 `./output/`에 저장합니다.

이 CSV들은 로컬 확인용이며 제출 ZIP에는 포함되지 않습니다.


In [ ]:
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

stage1_pred = baseline_inference.predict_stage1(SMOKE_DIR / "stage1", MODEL_DIR / "stage1")
stage2_pred = baseline_inference.predict_stage2(SMOKE_DIR / "stage2", MODEL_DIR / "stage2")
stage3_pred = baseline_inference.predict_stage3(SMOKE_DIR / "stage3", MODEL_DIR / "stage3")

predictions = {
    "stage1": stage1_pred,
    "stage2": stage2_pred,
    "stage3": stage3_pred,
}

for stage, frame in predictions.items():
    output_path = OUTPUT_DIR / f"{stage}_submission.csv"
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{stage}: {len(frame):,}행 -> {output_path}")
    display(frame.head())


## 9. 실제 제출용 `submit.zip` 생성 및 최종 검사

5번 단계에서 참가자가 생성한 `inference.py`, 학습된 모델 및 `requirements.txt`를 압축합니다. 이후 ZIP 내부의 `inference.py`를 다시 읽어 세 함수와 필수 모델 파일을 최종 검사합니다.


In [ ]:
import json
import zipfile
import ast

SUBMIT_PATH = ROOT / "submit.zip"

if not INFERENCE_PATH.is_file():
    raise FileNotFoundError(
        f"추론 파일이 없습니다: {INFERENCE_PATH}. 5번 inference.py 생성 단계를 먼저 실행해 주세요."
    )

inference_source = INFERENCE_PATH.read_text(encoding="utf-8")

# ZIP을 만들기 전에 실제 모듈 최상위에 Stage별 함수가 모두 있는지 검증합니다.
tree = ast.parse(inference_source, filename="inference.py")
defined_functions = {
    node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
required_functions = {"predict_stage1", "predict_stage2", "predict_stage3"}
missing_functions = sorted(required_functions - defined_functions)
if missing_functions:
    raise RuntimeError(
        "inference.py 생성 실패: 필수 추론 함수가 없습니다: " + str(missing_functions)
    )

if SUBMIT_PATH.exists():
    SUBMIT_PATH.unlink()

with zipfile.ZipFile(SUBMIT_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    archive.writestr("inference.py", inference_source)
    archive.write(ROOT / "requirements.txt", "requirements.txt")
    for model_path in sorted(MODEL_DIR.rglob("*")):
        if model_path.is_file():
            archive.write(model_path, model_path.relative_to(ROOT).as_posix())

with zipfile.ZipFile(SUBMIT_PATH) as archive:
    names = archive.namelist()
    zipped_inference = archive.read("inference.py").decode("utf-8")

required = {
    "inference.py",
    "requirements.txt",
    "model/stage1/best.pt",
    "model/stage2/best.pt",
    "model/stage2/resnet18-f37072fd.pth",
    "model/stage3/best.pt",
}
missing = sorted(required - set(names))
if missing:
    raise RuntimeError("제출 ZIP 필수 파일 누락: " + str(missing))

zipped_tree = ast.parse(zipped_inference, filename="submit.zip/inference.py")
zipped_functions = {
    node.name for node in zipped_tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
missing_in_zip = sorted(required_functions - zipped_functions)
if missing_in_zip:
    raise RuntimeError("ZIP 내부 inference.py 함수 누락: " + str(missing_in_zip))

print(f"생성 완료: {SUBMIT_PATH}")
print(f"압축 크기: {SUBMIT_PATH.stat().st_size / 1024**3:.3f} GB")
print("ZIP 내부 추론 함수:", sorted(required_functions))
print("ZIP 내부 파일:")
for name in names:
    print(" -", name)
